# 05 Limitations: Missing Data And Sample Reliability

This notebook now reads the current preprocessing artifacts produced by `src.reprocess_current_raw` and focuses on the remaining sample limitations after rebuilding the `2000-2023` panel from the new raw files.


## Setup

Load the clean panel and the export-side audit tables produced during preprocessing.


In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import OUTPUTS_DIR, PROCESSED_PANEL_FILE

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

clean_panel = pd.read_csv(PROCESSED_PANEL_FILE)
raw_input_audit = pd.read_csv(OUTPUTS_DIR / "raw_input_audit.csv")
variable_audit = pd.read_csv(OUTPUTS_DIR / "variable_audit.csv")
control_imputation_log = pd.read_csv(OUTPUTS_DIR / "control_imputation_log.csv")
coverage_by_country = pd.read_csv(OUTPUTS_DIR / "coverage_by_country.csv")
transformation_audit = pd.read_csv(OUTPUTS_DIR / "transformation_audit.csv")
review_flags = pd.read_csv(OUTPUTS_DIR / "review_flags.csv")


## Source Coverage

Start from the file-level audit so it is clear which source drives each remaining structural gap.


In [2]:
raw_input_audit


,file_name,role,years,countries,rows,notes
0,2000-2025-bm-ir-infl.xlsx,base_macro_panel,2000-2023,11,264,Uses WDI-style country-year rows directly from...
1,2000-2023-hc-pop.csv,human_capital_population_panel,2000-2023,10,240,"Melts row-by-variable yearly columns, uses onl..."
2,2000-2025-lending-deposit-tourism-arrivals.csv,supplementary_interest_tourism_panel,2000-2023,10,229,"Melts year columns and merges deposit rate, le..."


## Variable-Level Missingness

Rank variables by remaining missing share after the current preprocessing rules.


In [3]:
variable_audit.sort_values(["missing_rate", "variable"], ascending=[False, True]).reset_index(drop=True)


,variable,non_missing,total_rows,missing,missing_rate,recommended_handling
0,lending_interest_rate_pct,192,264,72,0.2727,processed_from_current_raw_inputs
1,real_interest_rate_pct,205,264,59,0.2235,processed_from_current_raw_inputs
2,deposit_interest_rate_pct,213,264,51,0.1932,processed_from_current_raw_inputs
3,broad_money_growth_pct,240,264,24,0.0909,processed_from_current_raw_inputs
4,hc_human_capital_index,240,264,24,0.0909,leave_missing_and_report_limitation
5,ln_population_total,240,264,24,0.0909,processed_from_current_raw_inputs
6,ln_tourism_arrivals,240,264,24,0.0909,processed_from_current_raw_inputs
7,trade_pct_gdp,240,264,24,0.0909,processed_from_current_raw_inputs
8,xr_dep_pct,253,264,11,0.0417,processed_from_current_raw_inputs
9,xr_dep_pct_winsorized,253,264,11,0.0417,processed_from_current_raw_inputs


## Imputation Log

Separate variables that were actually filled from variables that remain structurally incomplete.


In [4]:
control_imputation_log


,variable,handling_applied,missing_before,missing_after,filled_values
0,trade_pct_gdp,within_country_interpolate_then_edge_fill,31,24,7
1,inflation_gdp_deflator_pct,within_country_interpolate_then_edge_fill,0,0,0
2,ln_gdppc,within_country_interpolate_then_edge_fill,0,0,0
3,xr_dep_pct,within_country_interpolate_and_edge_fill_excep...,14,11,3
4,ln_population_total,within_country_interpolate_then_edge_fill,24,24,0
5,ln_tourism_arrivals,within_country_interpolate_then_edge_fill_keep...,64,24,40
6,hc_human_capital_index,leave_missing_and_report_limitation,24,24,0


## Country Coverage

Show how much usable information remains by country across the main variables carried into later notebooks.


In [5]:
coverage_by_country


,country,fdi_pct_gdp,broad_money_growth_pct,deposit_interest_rate_pct,real_interest_rate_pct,lending_interest_rate_pct,trade_pct_gdp,inflation_gdp_deflator_pct,ln_gdppc,xr_dep_pct,ln_tourism_arrivals,ln_population_total,hc_human_capital_index
0,Brunei Darussalam,24,24,21,24,24,24,24,24,23,24,24,24
1,Cambodia,24,24,24,0,0,24,24,24,23,24,24,24
2,Indonesia,24,24,24,24,24,24,24,24,23,24,24,24
3,Lao PDR,24,11,11,11,11,24,24,24,23,24,24,24
4,Malaysia,24,24,24,24,24,24,24,24,23,24,24,24
5,Myanmar,24,21,21,21,21,0,24,24,23,24,24,24
6,Philippines,24,23,20,20,20,24,24,24,23,24,24,24
7,Singapore,24,21,22,22,22,24,24,24,23,24,24,24
8,Thailand,24,24,22,22,22,24,24,24,23,24,24,24
9,Timor-Leste,20,21,0,13,0,24,24,24,23,0,0,0


## Remaining Gaps

Summarize which countries still have zero coverage for the variables that matter most for the expanded panel.


In [6]:
key_variables = [
    "deposit_interest_rate_pct",
    "real_interest_rate_pct",
    "lending_interest_rate_pct",
    "ln_tourism_arrivals",
    "hc_human_capital_index",
    "ln_population_total",
]

zero_coverage_rows = []
for variable in key_variables:
    zero_countries = coverage_by_country.loc[coverage_by_country[variable].eq(0), "country"].tolist()
    zero_coverage_rows.append(
        {
            "variable": variable,
            "countries_with_zero_coverage": ", ".join(zero_countries) if zero_countries else "",
            "zero_coverage_country_count": len(zero_countries),
        }
    )

pd.DataFrame(zero_coverage_rows)


,variable,countries_with_zero_coverage,zero_coverage_country_count
0,deposit_interest_rate_pct,Timor-Leste,1
1,real_interest_rate_pct,Cambodia,1
2,lending_interest_rate_pct,"Cambodia, Timor-Leste",2
3,ln_tourism_arrivals,Timor-Leste,1
4,hc_human_capital_index,Timor-Leste,1
5,ln_population_total,Timor-Leste,1


## Transformation And Review Audit

Keep the processing decisions visible for later writeup and diagnostics.


In [7]:
transformation_audit


,source_variable,transformed_variable,transformation,status,preferred_for_modeling,non_missing_source,non_missing_transformed
0,gdppc_current_usd,ln_gdppc,natural_log,created_from_current_macro_workbook,True,264.0000,264
1,official_exchange_rate_lcu_usd,xr_dep_pct,country_log_difference_x100,created_from_current_macro_workbook,True,261.0000,253
2,population_total,ln_population_total,natural_log,created_from_current_hc_pop_workbook,True,240.0000,240
3,hc_human_capital_index_raw_digits,hc_human_capital_index,direct_from_hc_series,created_from_current_hc_pop_workbook,True,NaN,240
4,2000-2025-lending-deposit-tourism-arrivals.csv,deposit_interest_rate_pct,country_series_merge_from_year_columns,created_from_additional_series_file,True,NaN,213
5,2000-2025-lending-deposit-tourism-arrivals.csv,lending_interest_rate_pct,country_series_merge_from_year_columns,created_from_additional_series_file,False,NaN,192
6,tourism_arrivals,ln_tourism_arrivals,natural_log,created_from_additional_series_file,False,200.0000,240


In [8]:
review_flags


,country,year,flag,value,note
0,Myanmar,2002,inflation_extreme_keep_and_review,41.5089,Observed value retained for main analysis.
1,Timor-Leste,2021,inflation_extreme_keep_and_review,59.0797,Observed value retained for main analysis.
2,Viet Nam,2010,inflation_extreme_keep_and_review,42.3033,Observed value retained for main analysis.
3,Myanmar,2012,xr_dep_pct_extreme_keep_and_review,476.7955,Observed value retained; winsorized version is...


In [9]:
pd.DataFrame(
    {
        "metric": ["rows", "countries", "first_year", "last_year"],
        "value": [
            len(clean_panel),
            clean_panel["country"].nunique(),
            int(clean_panel["year"].min()),
            int(clean_panel["year"].max()),
        ],
    }
)


,metric,value
0,rows,264
1,countries,11
2,first_year,2000
3,last_year,2023
